#### Scenario 1

An e-commerce company receives customer information from multiple sources.
Due to repeated registrations, the same customer can appear multiple times.

The data team needs to identify duplicate customers and keep one valid record.

#### Objective

- Identify duplicate customer records.
- Analyze duplicate patterns in customer data.
- Remove duplicate records based on business rules.
- Standardize email values for consistent data quality.
- Validate cleaned customer data.

#### Create Customer Data

In [0]:
data = [
    (101, "Alice", "Alice@gmail.com", "2026-01-01"),
    (102, "Bob", "bob@gmail.com", "2026-01-02"),
    (103, "Alice", "alice@gmail.com", "2026-01-05"),
    (104, "David", "david@gmail.com", "2026-01-07"),
    (105, "Bob", "BOB@gmail.com", "2026-01-09")
]

columns = [
    "customer_id",
    "customer_name",
    "email",
    "created_date"
]

customers_df = spark.createDataFrame(data,columns)
customers_df.show()

+-----------+-------------+---------------+------------+
|customer_id|customer_name|          email|created_date|
+-----------+-------------+---------------+------------+
|        101|        Alice|Alice@gmail.com|  2026-01-01|
|        102|          Bob|  bob@gmail.com|  2026-01-02|
|        103|        Alice|alice@gmail.com|  2026-01-05|
|        104|        David|david@gmail.com|  2026-01-07|
|        105|          Bob|  BOB@gmail.com|  2026-01-09|
+-----------+-------------+---------------+------------+



#### Create Temporary SQL View
Creates a temporary SQL view from the DataFrame.</br>
The view is available only within the active Spark session.

In [0]:
customers_df.createOrReplaceTempView("customers")

In [0]:
%sql

SELECT * FROM customers;

customer_id,customer_name,email,created_date
101,Alice,Alice@gmail.com,2026-01-01
102,Bob,bob@gmail.com,2026-01-02
103,Alice,alice@gmail.com,2026-01-05
104,David,david@gmail.com,2026-01-07
105,Bob,BOB@gmail.com,2026-01-09


#### 1.1 Identify duplicate customers | SQL

In [0]:
%sql
-- Identify duplicate customers by email count
 
SELECT LOWER(email) AS email, COUNT(*) AS duplicate_count
FROM customers
GROUP BY LOWER(email)
HAVING duplicate_count > 1;

email,duplicate_count
alice@gmail.com,2
bob@gmail.com,2


#### 1.2 Identify duplicate customers | PySpark

In [0]:
from pyspark.sql import functions as f

duplicate_customers = customers_df \
     .groupby(f.lower(f.col("email")).alias("email"))\
     .count()\
     .filter(f.col("count") > 1)\

duplicate_customers.show()

+---------------+-----+
|          email|count|
+---------------+-----+
|alice@gmail.com|    2|
|  bob@gmail.com|    2|
+---------------+-----+



#### 2.1 Understand Duplicate Patterns | SQL

In [0]:
%sql
-- Understand duplicate patterns by customer email

SELECT LOWER(email) AS email, COUNT(*) AS duplicate_count
FROM customers
GROUP BY LOWER(email)
ORDER BY duplicate_count DESC;

email,duplicate_count
alice@gmail.com,2
bob@gmail.com,2
david@gmail.com,1


#### 2.2 Understand Duplicate Patterns | PySpark


In [0]:
duplicate_patterns = customers_df \
    .groupBy(f.lower(f.col("email")).alias("email")) \
    .count()

duplicate_patterns.show()

+---------------+-----+
|          email|count|
+---------------+-----+
|alice@gmail.com|    2|
|  bob@gmail.com|    2|
|david@gmail.com|    1|
+---------------+-----+



#### 3.1 Remove Duplicate Records | SQL


In [0]:
%sql
-- CTE query: Assign row numbers to customer records to identify duplicates
WITH customer_duplicates AS 
(
SELECT customer_id, customer_name, email, created_date,
ROW_NUMBER() OVER (PARTITION BY LOWER(email) ORDER BY created_date DESC) AS rn
FROM customers
)

-- Main query: Filter and keep only the latest customer record
SELECT customer_id, customer_name, LOWER(email) AS email, created_date
FROM customer_duplicates
WHERE rn = 1;

customer_id,customer_name,email,created_date
103,Alice,alice@gmail.com,2026-01-05
105,Bob,bob@gmail.com,2026-01-09
104,David,david@gmail.com,2026-01-07


#### 3.2 Remove Duplicate Records | PySpark

##### Import Required Libraries

In [0]:
# Import PySpark SQL functions with alias f
from pyspark.sql import functions as f

# Import PySpark Window module with alias w
from pyspark.sql.window import Window as w

##### Define Window Specification

In [0]:
# Define Window specification

ws = w \
    .partitionBy(
        f.lower(f.col("email"))
    ) \
    .orderBy(
        f.col("created_date").desc()
    )

##### Apply Window Function

In [0]:
customers_clean_df = customers_df \
                     .withColumn("rn", f.row_number().over(ws))

customers_clean_df.show()

+-----------+-------------+---------------+------------+---+
|customer_id|customer_name|          email|created_date| rn|
+-----------+-------------+---------------+------------+---+
|        103|        Alice|alice@gmail.com|  2026-01-05|  1|
|        101|        Alice|Alice@gmail.com|  2026-01-01|  2|
|        105|          Bob|  BOB@gmail.com|  2026-01-09|  1|
|        102|          Bob|  bob@gmail.com|  2026-01-02|  2|
|        104|        David|david@gmail.com|  2026-01-07|  1|
+-----------+-------------+---------------+------------+---+



##### Remove Duplicate Records Based on Business Rules

In [0]:
# Keep only the latest customer record from each duplicate group
# Standardize email format after removing duplicates
clean_customers_df = customers_clean_df \
                    .filter(f.col("rn") == 1) \
                    .withColumn("email", f.lower(f.col("email"))) \
                    .drop("rn")
clean_customers_df.show()

+-----------+-------------+---------------+------------+
|customer_id|customer_name|          email|created_date|
+-----------+-------------+---------------+------------+
|        103|        Alice|alice@gmail.com|  2026-01-05|
|        105|          Bob|  bob@gmail.com|  2026-01-09|
|        104|        David|david@gmail.com|  2026-01-07|
+-----------+-------------+---------------+------------+



In [0]:
clean_customers_df.show()

+-----------+-------------+---------------+------------+
|customer_id|customer_name|          email|created_date|
+-----------+-------------+---------------+------------+
|        103|        Alice|alice@gmail.com|  2026-01-05|
|        105|          Bob|  bob@gmail.com|  2026-01-09|
|        104|        David|david@gmail.com|  2026-01-07|
+-----------+-------------+---------------+------------+



##### Final Validation

In [0]:
%sql
-- Validate that no duplicate emails exist after cleaning

WITH customer_duplicates AS (
SELECT customer_id, customer_name, LOWER(email) AS email, created_date,
ROW_NUMBER() OVER (PARTITION BY LOWER(email) ORDER BY created_date DESC) AS rn
FROM customers
)

SELECT email, COUNT(*) AS duplicate_count
FROM customer_duplicates WHERE rn = 1
GROUP BY email HAVING COUNT(*) > 1;

email,duplicate_count


In [0]:
# Check for duplicates after cleaning (PySpark)

customer_final_df = clean_customers_df \
                    .groupBy(f.lower(f.col("email")).alias("email")) \
                    .count()

customer_final_df.show()

+---------------+-----+
|          email|count|
+---------------+-----+
|alice@gmail.com|    1|
|  bob@gmail.com|    1|
|david@gmail.com|    1|
+---------------+-----+



#### Key Learnings

- Learned how to identify and remove duplicate records.
- Understood data cleaning techniques using SQL and PySpark.
- Applied business rules to maintain accurate customer data.
- Improved understanding of data validation after cleaning.